# KD 02 - ConvNeXt-Tiny (teacher) -> MobileNetV3-Small (student) - augmented dataset (Kaggle)



A deployment-shaped pair: ~28 M teacher into a ~2.5 M student meant for a phone or an edge box.

This notebook adds a small **(alpha, temperature) sweep**. alpha and T are the two knobs that
actually move KD results, and the best pair is dataset-dependent - a hard teacher (low T) passes
on confident labels, a soft teacher (high T) passes on the class-similarity structure
(`Leaf Blast` vs `Brown Spot` confusions), which is where the gain usually comes from.

The winner is chosen on **val**, which is what makes the test column an honest estimate of it.
Whether the winning (alpha, T) is the same on both datasets is itself a result worth reporting.
Each sweep run is 5 epochs, same as every other run in this folder.

## Running this on Kaggle

This notebook is **self-contained** - `kd_common`, `kd_models` and `kd_train` are inlined as
cells 1-3 instead of being imported from sibling `.py` files, because Kaggle notebooks cannot
import from a folder of scripts. The code is otherwise identical to the local `kd_aug/` version,
so results are comparable.

| step | what to do |
|---|---|
| dataset | **Add Input -> Datasets ->** your augmented rice-leaf dataset (`<root>/<class>/*.jpg`). Cell 0 already knows the `rice-lead-augmented` paths and otherwise searches `/kaggle/input`; set `DATA_DIR_OVERRIDE` to force a path. |
| accelerator | **Settings -> Accelerator -> GPU** (T4 x1 or P100). On CPU this will not finish. |
| internet | **Settings -> Internet -> ON** - torchvision downloads the pretrained backbones on first use. |
| run | Run All. Checkpoints and `results_*.json` are written to `/kaggle/working/checkpoints/`. |

Every teacher, assistant and student trains for **5 epochs**. Each Kaggle session is independent,
so this notebook trains its own teacher rather than reusing one from another notebook - if you
want to skip that, attach a previous run's output as an input dataset and copy the `.pth` into
`/kaggle/working/checkpoints/` before running.

## Setup

In [ ]:
# =============================================================================
# 0. Kaggle setup - paths, dataset discovery, environment
# =============================================================================
# HOW TO USE THIS NOTEBOOK ON KAGGLE
#   1. Add your augmented rice-leaf dataset:  Add Input -> Datasets -> pick yours.
#      It must be ImageFolder-shaped:  <root>/<class name>/*.jpg
#   2. Settings -> Accelerator -> GPU (T4 x1 or P100).
#   3. Settings -> Internet -> ON  (torchvision downloads the pretrained weights).
#   4. Run All.
#
# Set this if you want to name the folder yourself; otherwise KNOWN_PATHS is tried
# first and then /kaggle/input is searched automatically.
DATA_DIR_OVERRIDE = None

# Tried in order. Kaggle mounts the same dataset at different paths depending on
# how it was attached, so both shapes are listed.
KNOWN_PATHS = [
    "/kaggle/input/datasets/mahiacademia/rice-lead-augmented/Augmented Images",
    "/kaggle/input/rice-lead-augmented/Augmented Images",
    r"E:\riceleaf\Augmented Images",          # running this notebook locally
]

import os
import sys

ON_KAGGLE = os.path.isdir("/kaggle/input")

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def find_dataset_root(base="/kaggle/input", max_depth=8):
    """Best-scoring directory whose subfolders are class folders of images.

    Kaggle nests datasets to different depths (/kaggle/input/<slug>/... but also
    /kaggle/input/datasets/<owner>/<slug>/...), so the search goes deep and ranks
    candidates by how many class folders and images they hold.
    """
    if not os.path.isdir(base):
        raise FileNotFoundError(
            "no /kaggle/input - set DATA_DIR_OVERRIDE to your dataset path")

    candidates = []
    base_depth = base.rstrip("/").count("/")
    for dirpath, dirnames, _ in os.walk(base):
        depth = dirpath.rstrip("/").count("/") - base_depth

        # score THIS directory before deciding whether to descend further
        if len(dirnames) >= 2:
            n_class_dirs, n_images = 0, 0
            for d in dirnames:
                try:
                    entries = os.listdir(os.path.join(dirpath, d))
                except OSError:
                    continue
                hits = [e for e in entries if e.lower().endswith(IMG_EXT)]
                if hits:
                    n_class_dirs += 1
                    n_images += len(hits)
            if n_class_dirs >= 2:
                candidates.append((n_class_dirs, n_images, dirpath))
                dirnames[:] = []          # class folders - do not descend into them
                continue

        if depth >= max_depth:
            dirnames[:] = []

    if not candidates:
        raise FileNotFoundError(
            "found no <root>/<class>/*.jpg layout under %s - "
            "attach the dataset or set DATA_DIR_OVERRIDE" % base)

    candidates.sort(key=lambda t: (-t[0], -t[1]))
    return candidates[0][2]


_DATA_DIR = None
if DATA_DIR_OVERRIDE:
    _DATA_DIR = DATA_DIR_OVERRIDE
else:
    for _p in KNOWN_PATHS:
        if os.path.isdir(_p):
            _DATA_DIR = _p
            break
    if _DATA_DIR is None and ON_KAGGLE:
        _DATA_DIR = find_dataset_root()

if not _DATA_DIR or not os.path.isdir(_DATA_DIR):
    raise FileNotFoundError(
        "dataset not found (%s) - attach it, or set DATA_DIR_OVERRIDE" % _DATA_DIR)

_CKPT_DIR = "/kaggle/working/checkpoints" if ON_KAGGLE else os.path.join(os.getcwd(), "checkpoints")

_CLASSES = sorted(d for d in os.listdir(_DATA_DIR)
                  if os.path.isdir(os.path.join(_DATA_DIR, d)))
_N_IMG = sum(len([e for e in os.listdir(os.path.join(_DATA_DIR, c))
                  if e.lower().endswith(IMG_EXT)]) for c in _CLASSES)

print("dataset     ->", _DATA_DIR)
print("classes (%d) -> %s" % (len(_CLASSES), _CLASSES))
print("images      ->", _N_IMG)
print("checkpoints ->", _CKPT_DIR)
assert len(_CLASSES) >= 2, "expected class subfolders under %s" % _DATA_DIR

In [ ]:
# =============================================================================
# 1. kd_common - data, model factory, KD/hint losses, eval, plots
# =============================================================================
"""
Shared helpers for the rice-leaf knowledge-distillation experiments.

KAGGLE EDITION - inlined, no sibling .py files. Identical to the local
kd_aug/kd_common.py except for the CONFIG block (paths + dataloader workers).

Dataset layout expected (ImageFolder style):
    <kaggle input root>/<class name>/*.jpg
"""

import os
import time
import json
import random

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision
from torchvision import datasets, transforms

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix)

import matplotlib.pyplot as plt
import seaborn as sns


# --------------------------------------------------------------------------- #
# CONFIG
# --------------------------------------------------------------------------- #
DATASET_DIR = _DATA_DIR          # set in the Kaggle setup cell above
CKPT_DIR = _CKPT_DIR
IMG_SIZE = 224
BATCH_SIZE = 16
VAL_SPLIT = 0.20         # 60 / 20 / 20 train / val / test, stratified
TEST_SPLIT = 0.20        # val picks the checkpoint; test is touched once, at the end
NUM_WORKERS = 2 if ON_KAGGLE else 0      # Kaggle is Linux, workers are safe there
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(CKPT_DIR, exist_ok=True)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# --------------------------------------------------------------------------- #
# DATA
# --------------------------------------------------------------------------- #
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def train_transform(img_size=IMG_SIZE, aug="medium"):
    """Augmentation strength.

    light  - for SHORT runs (<= 5-8 epochs). Heavy augmentation needs epochs to pay
             off; inside 5 epochs it just costs accuracy, and it also blurs the
             teacher's soft targets. Use this when chasing a fast accuracy target.
    medium - the default used by notebooks 01-06.
    strong - only worth it for long runs (20+ epochs).
    """
    if aug == "light":
        return transforms.Compose([
            transforms.Resize((img_size + 16, img_size + 16)),
            transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0), ratio=(0.9, 1.11)),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.2),
            transforms.RandomRotation(15),
            transforms.ColorJitter(0.15, 0.15, 0.15, 0.02),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    if aug == "strong":
        return transforms.Compose([
            transforms.Resize((img_size + 48, img_size + 48)),
            transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.5),
            transforms.RandomRotation(45),
            transforms.ColorJitter(0.35, 0.35, 0.35, 0.1),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)),
        ])
    return transforms.Compose([
        transforms.Resize((img_size + 32, img_size + 32)),
        transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0), ratio=(0.8, 1.25)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.3),
        transforms.RandomRotation(30),
        transforms.ColorJitter(0.25, 0.25, 0.25, 0.05),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
    ])


def eval_transform(img_size=IMG_SIZE):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class RiceSubset(Dataset):
    """A (path, label) list plus its own transform. Avoids the shared-transform
    problem you get when wrapping a single ImageFolder in two Subsets."""

    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label


def build_data(img_size=IMG_SIZE, batch_size=BATCH_SIZE, val_split=VAL_SPLIT,
               test_split=TEST_SPLIT, seed=SEED, balanced_sampler=True, verbose=True,
               aug="medium", sampler_power=1.0, weight_power=0.5):
    """Stratified train/val/test split -> DataLoaders + class metadata.

    Default 60 / 20 / 20. Both splits are stratified and driven by the same seed,
    so the three sets are identical across every notebook and every model.

    val  : evaluated each epoch; picks the best checkpoint (and shows the curves).
    test : held out, evaluated once at the end - the number to report.

    sampler_power : how hard the sampler rebalances classes.
        1.0 = fully balanced (every class equally likely). On a severely skewed
              set that means showing the smallest class's few images many times
              per epoch - the model memorises them and OVERALL ACCURACY DROPS.
              0.0 = natural frequency. 0.3-0.5 is the sweet spot when the metric
              you report is accuracy. This dataset is only mildly skewed
              (300-790 per class), so the setting matters less than it does in
              kd/ - it is kept identical for comparability.
    weight_power  : same idea for the cross-entropy class weights (0 = unweighted).
    """
    base = datasets.ImageFolder(DATASET_DIR)
    class_names = base.classes
    num_classes = len(class_names)
    samples = base.samples
    labels = np.array([lbl for _, lbl in samples])

    if not 0.0 < val_split + test_split < 1.0:
        raise ValueError("val_split + test_split must be in (0, 1)")

    # two stages: peel off test first, then carve val out of what is left, sized
    # so val_split/test_split stay fractions of the WHOLE dataset
    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=test_split, random_state=seed)
    trainval_idx, test_idx = next(sss_test.split(np.zeros(len(labels)), labels))

    rel_val = val_split / (1.0 - test_split)
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=rel_val, random_state=seed)
    tv_labels = labels[trainval_idx]
    rel_train, rel_val_idx = next(sss_val.split(np.zeros(len(tv_labels)), tv_labels))
    train_idx = trainval_idx[rel_train]
    val_idx = trainval_idx[rel_val_idx]

    train_ds = RiceSubset([samples[i] for i in train_idx], train_transform(img_size, aug))
    val_ds = RiceSubset([samples[i] for i in val_idx], eval_transform(img_size))
    test_ds = RiceSubset([samples[i] for i in test_idx], eval_transform(img_size))

    train_labels = labels[train_idx]
    counts = np.bincount(train_labels, minlength=num_classes).astype(np.float64)
    counts[counts == 0] = 1.0

    if balanced_sampler and sampler_power > 0:
        per_class_w = (1.0 / counts) ** sampler_power
        sample_w = per_class_w[train_labels]
        sampler = WeightedRandomSampler(torch.as_tensor(sample_w, dtype=torch.double),
                                        num_samples=len(sample_w), replacement=True)
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    else:
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

    # class weights on the CE term - the sampler already does part of the balancing
    cw = (counts.sum() / (num_classes * counts)) ** weight_power
    class_weights = torch.tensor(cw, dtype=torch.float, device=DEVICE)

    if verbose:
        total = len(train_ds) + len(val_ds) + len(test_ds)
        print("classes (%d): %s" % (num_classes, class_names))
        print("train: %d (%.0f%%) | val: %d (%.0f%%) | test: %d (%.0f%%) | total %d"
              % (len(train_ds), 100.0 * len(train_ds) / total,
                 len(val_ds), 100.0 * len(val_ds) / total,
                 len(test_ds), 100.0 * len(test_ds) / total, total))
        print("img %d | bs %d | aug %s | sampler_power %.2f"
              % (img_size, batch_size, aug, sampler_power))
        print("train per-class counts:",
              {class_names[i]: int(counts[i]) for i in range(num_classes)})
        test_counts = np.bincount(labels[test_idx], minlength=num_classes)
        print("test  per-class counts:",
              {class_names[i]: int(test_counts[i]) for i in range(num_classes)})
        print("class weights:", np.round(cw, 3))

    return {
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "class_names": class_names,
        "num_classes": num_classes,
        "class_weights": class_weights,
    }


# --------------------------------------------------------------------------- #
# MODELS
# --------------------------------------------------------------------------- #
def _replace_head(model, num_classes, dropout=0.0):
    """Swap the final Linear of any torchvision classifier for a fresh one."""
    for attr in ("heads", "head", "classifier", "fc"):
        if not hasattr(model, attr):
            continue
        mod = getattr(model, attr)

        if isinstance(mod, nn.Linear):
            new = nn.Linear(mod.in_features, num_classes)
            setattr(model, attr, nn.Sequential(nn.Dropout(dropout), new) if dropout else new)
            return model

        if isinstance(mod, nn.Sequential):
            lin_idx = [i for i, m in enumerate(mod) if isinstance(m, nn.Linear)]
            if lin_idx:
                i = lin_idx[-1]
                mod[i] = nn.Linear(mod[i].in_features, num_classes)
                return model

        if hasattr(mod, "head") and isinstance(mod.head, nn.Linear):   # ViT: model.heads.head
            mod.head = nn.Linear(mod.head.in_features, num_classes)
            return model

    raise ValueError("Could not locate a classifier head on %s" % type(model).__name__)


def build_model(name, num_classes, pretrained=True, dropout=0.0):
    """Any torchvision classification model by name, head resized to num_classes.

    Useful names: vit_b_16, swin_t, convnext_tiny, efficientnet_b3, efficientnet_b0,
    resnet50, resnet34, resnet18, mobilenet_v3_small, mobilenet_v2,
    shufflenet_v2_x1_0, densenet121, regnet_y_400mf, mnasnet1_0, squeezenet1_1.
    """
    weights = "DEFAULT" if pretrained else None
    model = torchvision.models.get_model(name, weights=weights)
    if name.startswith("squeezenet"):
        model.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=1)
        model.num_classes = num_classes
    else:
        _replace_head(model, num_classes, dropout=dropout)
    model.model_name = name
    return model


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def freeze_backbone(model, trainable_keywords=("head", "classifier", "fc")):
    for n, p in model.named_parameters():
        p.requires_grad = any(k in n for k in trainable_keywords)
    return model


# Keep this list conservative: "attn"/"proj" would also match pretrained ViT
# (self_attention.in_proj_weight) and Swin (attn.*) parameters and hand them the
# high head learning rate. Custom modules should be NAMED head_* instead.
HEAD_KEYS = ("head", "classifier", "fc")


def strip_head(model):
    """Replace the final Linear with Identity so the model outputs pooled FEATURES.
    Returns (model, feature_dim). Used to build custom multi-backbone models."""
    for attr in ("heads", "head", "classifier", "fc"):
        if not hasattr(model, attr):
            continue
        mod = getattr(model, attr)
        if isinstance(mod, nn.Linear):
            dim = mod.in_features
            setattr(model, attr, nn.Identity())
            return model, dim
        if isinstance(mod, nn.Sequential):
            lin_idx = [i for i, m in enumerate(mod) if isinstance(m, nn.Linear)]
            if lin_idx:
                i = lin_idx[-1]
                dim = mod[i].in_features
                mod[i] = nn.Identity()
                return model, dim
        if hasattr(mod, "head") and isinstance(mod.head, nn.Linear):
            dim = mod.head.in_features
            mod.head = nn.Identity()
            return model, dim
    raise ValueError("Could not locate a classifier head on %s" % type(model).__name__)


def param_groups(model, head_lr, backbone_lr, weight_decay=1e-4, head_keys=HEAD_KEYS):
    """Discriminative learning rates: fresh head fast, pretrained backbone slow."""
    head, backbone = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head if any(k in n for k in head_keys) else backbone).append(p)
    groups = []
    if head:
        groups.append({"params": head, "lr": head_lr, "weight_decay": weight_decay})
    if backbone:
        groups.append({"params": backbone, "lr": backbone_lr, "weight_decay": weight_decay})
    return groups


@torch.no_grad()
def measure_latency(model, img_size=IMG_SIZE, device=DEVICE, runs=50, warmup=10, batch=1):
    """Milliseconds per image at batch size 1 (inference)."""
    model = model.to(device).eval()
    x = torch.randn(batch, 3, img_size, img_size, device=device)
    for _ in range(warmup):
        model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs):
        model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs * 1000.0


# --------------------------------------------------------------------------- #
# LOSSES
# --------------------------------------------------------------------------- #
def kd_loss(student_logits, teacher_probs, targets, ce, alpha=0.7, T=4.0,
            targets_b=None, lam=1.0):
    """Hinton KD: (1-alpha) * CE(student, y) + alpha * T^2 * KL(student_T || teacher_T).

    teacher_probs: teacher probabilities ALREADY softened at temperature T.
    targets_b/lam: set when the batch was mixed up (lam=1 means no mixup).
    Returns (total, hard_part, soft_part).
    """
    hard = ce(student_logits, targets)
    if targets_b is not None and lam < 1.0:
        hard = lam * hard + (1.0 - lam) * ce(student_logits, targets_b)
    s_log = F.log_softmax(student_logits / T, dim=1)
    soft = F.kl_div(s_log, teacher_probs, reduction="batchmean") * (T ** 2)
    return (1.0 - alpha) * hard + alpha * soft, hard.detach(), soft.detach()


class DistillationLoss(nn.Module):
    """nn.Module wrapper around kd_loss(), for use outside the training helpers."""

    def __init__(self, alpha=0.7, T=4.0, class_weights=None, label_smoothing=0.05):
        super().__init__()
        self.alpha = alpha
        self.T = T
        self.ce = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)

    def forward(self, student_logits, teacher_probs, targets):
        return kd_loss(student_logits, teacher_probs, targets, self.ce,
                       alpha=self.alpha, T=self.T)


@torch.no_grad()
def teacher_soft_targets(teachers, inputs, T):
    """One teacher or a list of them -> averaged softened probabilities."""
    if not isinstance(teachers, (list, tuple)):
        teachers = [teachers]
    probs = None
    for t in teachers:
        t.eval()
        p = F.softmax(t(inputs) / T, dim=1)
        probs = p if probs is None else probs + p
    return probs / len(teachers)


class FeatureCatcher:
    """Forward hook that stores a module's output (for hint / FitNets losses)."""

    def __init__(self, module):
        self.output = None
        self.handle = module.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        self.output = out

    def pooled(self):
        """Whatever the module emitted -> a (B, C) vector."""
        f = self.output
        if isinstance(f, (list, tuple)):
            f = f[0]
        if f.dim() == 4:
            if f.shape[1] > f.shape[-1]:       # B,C,H,W (plain CNN)
                f = F.adaptive_avg_pool2d(f, 1).flatten(1)
            else:                              # B,H,W,C (ConvNeXt / Swin, channels-last)
                f = f.mean(dim=(1, 2))
        elif f.dim() == 3:                     # B,N,C (transformer tokens)
            f = f.mean(dim=1)
        return f

    def close(self):
        self.handle.remove()


def make_projector(student, teacher, s_module, t_module, img_size=IMG_SIZE, device=DEVICE):
    """Dummy forward to learn feature widths, then build a student->teacher projector."""
    sc, tc = FeatureCatcher(s_module), FeatureCatcher(t_module)
    student.eval()
    teacher.eval()
    with torch.no_grad():
        x = torch.randn(2, 3, img_size, img_size, device=device)
        student(x)
        teacher(x)
        s_dim, t_dim = sc.pooled().shape[1], tc.pooled().shape[1]
    sc.close()
    tc.close()
    print("hint dims: student %d -> teacher %d" % (s_dim, t_dim))
    return nn.Sequential(nn.Linear(s_dim, t_dim), nn.BatchNorm1d(t_dim)).to(device)


def hint_loss(s_feat, t_feat, projector):
    """Normalised MSE between projected student features and teacher features."""
    s = projector(s_feat)
    return F.mse_loss(F.normalize(s, dim=1), F.normalize(t_feat.detach(), dim=1))


def mixup_batch(x, y, alpha=0.2):
    """Returns mixed inputs, the permutation index and lambda."""
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1.0 - lam) * x[idx], idx, lam


# --------------------------------------------------------------------------- #
# EVAL / REPORTING
# --------------------------------------------------------------------------- #
@torch.no_grad()
def evaluate(model, loader, criterion=None, device=DEVICE, desc="eval", tta=False):
    """tta=True averages the softmax over the 4 flips (identity, h, v, hv).
    Costs 4x inference, typically worth +1-2 accuracy points. Report it as TTA."""
    model.eval().to(device)
    preds, targets, total_loss = [], [], 0.0
    for x, y in tqdm(loader, desc=desc, leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        if tta:
            views = [x, torch.flip(x, [3]), torch.flip(x, [2]), torch.flip(x, [2, 3])]
            probs = torch.stack([F.softmax(model(v), dim=1) for v in views]).mean(0)
            out = torch.log(probs.clamp_min(1e-9))
        else:
            out = model(x)
        if criterion is not None:
            total_loss += criterion(out, y).item() * x.size(0)
        preds.append(out.argmax(1).cpu())
        targets.append(y.cpu())
    preds = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()
    return {
        "acc": accuracy_score(targets, preds),
        "f1": f1_score(targets, preds, average="weighted"),
        "loss": total_loss / len(loader.dataset) if criterion is not None else float("nan"),
        "preds": preds,
        "targets": targets,
    }


def evaluate_splits(model, data, desc="model", tta=False, criterion=None):
    """Evaluate on BOTH the val and the test loader -> (val_res, test_res).

    val selected the checkpoint, so it is optimistic; test was never looked at
    during training and is the number to quote.
    """
    val_res = evaluate(model, data["val_loader"], criterion=criterion,
                       desc=desc + "/val", tta=tta)
    test_res = evaluate(model, data["test_loader"], criterion=criterion,
                        desc=desc + "/test", tta=tta)
    return val_res, test_res


def report_both(res_val, res_test, class_names, title="model"):
    """Headline val/test lines, then the full per-class breakdown on TEST."""
    print("\n=== %s ===" % title)
    print("val   acc %.4f | weighted F1 %.4f" % (res_val["acc"], res_val["f1"]))
    print("test  acc %.4f | weighted F1 %.4f  <- report this one"
          % (res_test["acc"], res_test["f1"]))
    print("val - test gap: %+.4f acc (large positive = val-selected overfitting)"
          % (res_val["acc"] - res_test["acc"]))
    print(classification_report(res_test["targets"], res_test["preds"],
                                target_names=class_names, digits=4, zero_division=0))


def report(res, class_names, title="model"):
    print("\n=== %s ===" % title)
    print("accuracy %.4f | weighted F1 %.4f" % (res["acc"], res["f1"]))
    print(classification_report(res["targets"], res["preds"],
                                target_names=class_names, digits=4, zero_division=0))


def plot_confusion(res, class_names, title="Confusion matrix"):
    cm = confusion_matrix(res["targets"], res["preds"])
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_histories(histories, keys=("val_acc", "val_f1")):
    """histories: {label: history dict} -> one subplot per key."""
    fig, axes = plt.subplots(1, len(keys), figsize=(6 * len(keys), 4))
    axes = np.atleast_1d(axes)
    for ax, key in zip(axes, keys):
        for label, h in histories.items():
            if key in h and len(h[key]):
                ax.plot(range(1, len(h[key]) + 1), h[key], marker="o", label=label)
        ax.set_xlabel("epoch")
        ax.set_ylabel(key)
        ax.set_title(key)
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()


def summarize(rows, img_size=IMG_SIZE):
    """rows: [{"name", "model", "result": val_res, "test": test_res}] -> comparison table.

    "test" is optional; when any row carries one the table gains test columns, so
    val and test sit side by side and the val-to-test drop is visible per model.
    """
    has_test = any(r.get("test") is not None for r in rows)
    if has_test:
        header = "%-34s%11s%8s%9s%8s%10s%9s" % ("model", "params(M)", "ms/img",
                                                "val_acc", "val_F1", "test_acc", "test_F1")
    else:
        header = "%-34s%11s%9s%9s%9s" % ("model", "params(M)", "ms/img", "acc", "F1")
    print(header)
    print("-" * len(header))
    out = []
    for r in rows:
        p = count_params(r["model"]) / 1e6
        ms = measure_latency(r["model"], img_size=img_size)
        v, t = r["result"], r.get("test")
        rec = {"name": r["name"], "params_m": p, "ms_per_img": ms,
               "val_acc": v["acc"], "val_f1": v["f1"],
               "acc": v["acc"], "f1": v["f1"]}          # legacy keys, kept
        if has_test:
            if t is None:
                print("%-34s%11.2f%8.2f%9.4f%8.4f%10s%9s"
                      % (r["name"], p, ms, v["acc"], v["f1"], "-", "-"))
            else:
                rec["test_acc"], rec["test_f1"] = t["acc"], t["f1"]
                print("%-34s%11.2f%8.2f%9.4f%8.4f%10.4f%9.4f"
                      % (r["name"], p, ms, v["acc"], v["f1"], t["acc"], t["f1"]))
        else:
            print("%-34s%11.2f%9.2f%9.4f%9.4f" % (r["name"], p, ms, v["acc"], v["f1"]))
        out.append(rec)
    return out


def save_results(tag, payload):
    path = os.path.join(CKPT_DIR, "results_%s.json" % tag)
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    print("saved ->", path)

In [ ]:
# =============================================================================
# 3. kd_train - train_supervised() and train_kd()
# =============================================================================
"""
Training loops shared by all the knowledge-distillation notebooks.

    train_supervised(...)  -> plain cross-entropy fine-tuning (teachers + student baselines)
    train_kd(...)          -> distillation: logit KD (+ optional feature hint, + optional mixup)

Both return a `history` dict and leave the model holding the best-F1 weights.
"""

import os
import copy
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# CKPT_DIR, DEVICE, IMG_SIZE, evaluate, param_groups, teacher_soft_targets,
# kd_loss, FeatureCatcher, hint_loss and mixup_batch come from the kd_common cell.


def _new_history():
    return {"train_loss": [], "train_acc": [], "val_loss": [],
            "val_acc": [], "val_f1": [], "lr": []}


def _ckpt_path(name):
    return os.path.join(CKPT_DIR, name if name.endswith(".pth") else name + ".pth")


# --------------------------------------------------------------------------- #
# 1. PLAIN SUPERVISED FINE-TUNING (teachers and student baselines)
# --------------------------------------------------------------------------- #
def train_supervised(model, data, epochs=3, head_lr=1e-3, backbone_lr=1e-4,
                     weight_decay=1e-4, label_smoothing=0.05, mixup_alpha=0.0,
                     ckpt_name=None, load_if_exists=False, use_amp=True,
                     device=DEVICE, tag="model", pct_start=0.3):
    """Standard fine-tuning. Set load_if_exists=True to reuse a cached teacher."""
    model = model.to(device)
    ckpt = _ckpt_path(ckpt_name) if ckpt_name else None

    if load_if_exists and ckpt and os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=device))
        res = evaluate(model, data["val_loader"], device=device, desc="cached " + tag)
        print("loaded cached %s from %s  ->  acc %.4f | F1 %.4f"
              % (tag, ckpt, res["acc"], res["f1"]))
        h = _new_history()
        h["val_acc"].append(res["acc"])
        h["val_f1"].append(res["f1"])
        return h

    ce = nn.CrossEntropyLoss(weight=data["class_weights"], label_smoothing=label_smoothing)
    opt = torch.optim.AdamW(param_groups(model, head_lr, backbone_lr, weight_decay))
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[g["lr"] for g in opt.param_groups],
        total_steps=epochs * len(data["train_loader"]), pct_start=pct_start)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and device.type == "cuda")

    history = _new_history()
    best_f1, best_state = -1.0, None
    t_start = time.time()

    for epoch in range(epochs):
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        bar = tqdm(data["train_loader"], desc="%s | epoch %d/%d" % (tag, epoch + 1, epochs))
        for x, y in bar:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
                if mixup_alpha > 0:
                    xm, idx, lam = mixup_batch(x, y, mixup_alpha)
                    out = model(xm)
                    loss = lam * ce(out, y) + (1 - lam) * ce(out, y[idx])
                else:
                    out = model(x)
                    loss = ce(out, y)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()

            run_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
            bar.set_postfix(loss=run_loss / total, acc=correct / total)

        val = evaluate(model, data["val_loader"], criterion=ce, device=device,
                       desc="%s val" % tag)
        history["train_loss"].append(run_loss / total)
        history["train_acc"].append(correct / total)
        history["val_loss"].append(val["loss"])
        history["val_acc"].append(val["acc"])
        history["val_f1"].append(val["f1"])
        history["lr"].append(opt.param_groups[0]["lr"])
        print("%s | epoch %d: train_loss %.4f train_acc %.4f | val_loss %.4f val_acc %.4f val_f1 %.4f"
              % (tag, epoch + 1, history["train_loss"][-1], history["train_acc"][-1],
                 val["loss"], val["acc"], val["f1"]))

        if val["f1"] > best_f1:
            best_f1 = val["f1"]
            best_state = copy.deepcopy(model.state_dict())
            if ckpt:
                torch.save(best_state, ckpt)

    if best_state is not None:
        model.load_state_dict(best_state)
    history["best_f1"] = best_f1
    history["minutes"] = (time.time() - t_start) / 60.0
    print("%s done in %.1f min | best val F1 %.4f" % (tag, history["minutes"], best_f1))
    return history


# --------------------------------------------------------------------------- #
# 2. KNOWLEDGE DISTILLATION
# --------------------------------------------------------------------------- #
def train_kd(student, teachers, data, epochs=5, alpha=0.7, T=4.0,
             head_lr=1e-3, backbone_lr=3e-4, weight_decay=1e-4,
             label_smoothing=0.05, mixup_alpha=0.0,
             beta=0.0, student_feat_module=None, teacher_feat_module=None, projector=None,
             ckpt_name=None, use_amp=True, device=DEVICE, tag="student-KD", verbose=True,
             pct_start=0.3):
    """Distil `teachers` (one model or a list -> ensemble) into `student`.

    alpha : weight on the soft (teacher) term; (1-alpha) goes to the hard CE term
    T     : softmax temperature
    beta  : weight on the feature-hint term (needs the two *_feat_module args)
    """
    student = student.to(device)
    if not isinstance(teachers, (list, tuple)):
        teachers = [teachers]
    for t in teachers:
        t.to(device).eval()
        for p in t.parameters():
            p.requires_grad = False

    use_hint = beta > 0 and student_feat_module is not None and teacher_feat_module is not None
    s_catch = t_catch = None
    if use_hint:
        s_catch = FeatureCatcher(student_feat_module)
        t_catch = FeatureCatcher(teacher_feat_module)
        if projector is None:
            raise ValueError("beta > 0 requires a projector (see kd_common.make_projector)")

    ce = nn.CrossEntropyLoss(weight=data["class_weights"], label_smoothing=label_smoothing)
    groups = param_groups(student, head_lr, backbone_lr, weight_decay)
    if use_hint:
        groups.append({"params": list(projector.parameters()), "lr": head_lr,
                       "weight_decay": weight_decay})
    opt = torch.optim.AdamW(groups)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[g["lr"] for g in opt.param_groups],
        total_steps=epochs * len(data["train_loader"]), pct_start=pct_start)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and device.type == "cuda")

    history = _new_history()
    history["hard_loss"], history["soft_loss"], history["hint_loss"] = [], [], []
    ckpt = _ckpt_path(ckpt_name) if ckpt_name else None
    best_f1, best_state = -1.0, None
    t_start = time.time()

    for epoch in range(epochs):
        student.train()
        run_loss = run_hard = run_soft = run_hint = 0.0
        correct, total = 0, 0
        bar = tqdm(data["train_loader"], desc="%s | epoch %d/%d" % (tag, epoch + 1, epochs))

        for x, y in bar:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)

            y_b, lam = None, 1.0
            x_in = x
            if mixup_alpha > 0:
                x_in, idx, lam = mixup_batch(x, y, mixup_alpha)
                y_b = y[idx]

            with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
                # teacher sees exactly what the student sees
                t_probs = teacher_soft_targets(teachers, x_in, T)
                out = student(x_in)
                loss, hard, soft = kd_loss(out, t_probs, y, ce, alpha=alpha, T=T,
                                           targets_b=y_b, lam=lam)
                hint = torch.zeros((), device=device)
                if use_hint:
                    hint = hint_loss(s_catch.pooled(), t_catch.pooled(), projector)
                    loss = loss + beta * hint

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()

            bs = x.size(0)
            run_loss += loss.item() * bs
            run_hard += hard.item() * bs
            run_soft += soft.item() * bs
            run_hint += float(hint) * bs
            correct += (out.argmax(1) == y).sum().item()
            total += bs
            bar.set_postfix(loss=run_loss / total, hard=run_hard / total, soft=run_soft / total)

        val = evaluate(student, data["val_loader"], criterion=ce, device=device,
                       desc="%s val" % tag)
        history["train_loss"].append(run_loss / total)
        history["train_acc"].append(correct / total)
        history["hard_loss"].append(run_hard / total)
        history["soft_loss"].append(run_soft / total)
        history["hint_loss"].append(run_hint / total)
        history["val_loss"].append(val["loss"])
        history["val_acc"].append(val["acc"])
        history["val_f1"].append(val["f1"])
        history["lr"].append(opt.param_groups[0]["lr"])

        if verbose:
            print("%s | epoch %d: loss %.4f (hard %.4f soft %.4f hint %.4f) | val_acc %.4f val_f1 %.4f"
                  % (tag, epoch + 1, history["train_loss"][-1], history["hard_loss"][-1],
                     history["soft_loss"][-1], history["hint_loss"][-1], val["acc"], val["f1"]))

        if val["f1"] > best_f1:
            best_f1 = val["f1"]
            best_state = copy.deepcopy(student.state_dict())
            if ckpt:
                torch.save(best_state, ckpt)

    if s_catch is not None:
        s_catch.close()
        t_catch.close()
    if best_state is not None:
        student.load_state_dict(best_state)
    history["best_f1"] = best_f1
    history["minutes"] = (time.time() - t_start) / 60.0
    print("%s done in %.1f min | best val F1 %.4f" % (tag, history["minutes"], best_f1))
    return history


# --------------------------------------------------------------------------- #
# 3. AGREEMENT / FIDELITY - how much of the teacher did the student actually copy?
# --------------------------------------------------------------------------- #
@torch.no_grad()
def teacher_student_agreement(student, teachers, loader, device=DEVICE):
    """Fraction of val images where student and teacher predict the same class."""
    if not isinstance(teachers, (list, tuple)):
        teachers = [teachers]
    student.eval().to(device)
    for t in teachers:
        t.eval().to(device)
    same, n = 0, 0
    for x, _ in tqdm(loader, desc="agreement", leave=False):
        x = x.to(device)
        s_pred = student(x).argmax(1)
        t_probs = teacher_soft_targets(teachers, x, T=1.0)
        same += (s_pred == t_probs.argmax(1)).sum().item()
        n += x.size(0)
    return same / n

In [ ]:
# =============================================================================
# 4. session
# =============================================================================
set_seed(SEED)
print("device:", DEVICE, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("dataset     ->", DATASET_DIR)
print("checkpoints ->", CKPT_DIR)
print("workers:", NUM_WORKERS, "| img", IMG_SIZE, "| default batch", BATCH_SIZE)

---

### Train / val / test

`build_data()` makes a **60 / 20 / 20** stratified split from one seed, so all three sets are
identical across every notebook and every model in this folder.

| set | share | used for |
|---|---|---|
| train | 60% | fitting |
| val | 20% | evaluated every epoch; **picks the best checkpoint**, drives the curves |
| test | 20% | held out, evaluated **once at the end** - the number to report |

Because the checkpoint is chosen on val, val is optimistic by construction. Every table below
prints val and test side by side, and `report_both()` prints the val-to-test gap, so the size of
that optimism is visible rather than hidden. Quote the **test** column in the writeup.

In [ ]:
# 60/20/20 stratified split, class-balanced sampler, mild sqrt class weights.
# Lower BATCH_SIZE in kd_common.py (or here) if you hit CUDA OOM.
data = build_data(img_size=IMG_SIZE, batch_size=BATCH_SIZE)
class_names, num_classes = data["class_names"], data["num_classes"]

In [ ]:
TEACHER_NAME = "convnext_tiny"
STUDENT_NAME = "mobilenet_v3_small"

TEACHER_EPOCHS = 5
STUDENT_EPOCHS = 5       # per sweep run
TAG = "kd02_convnext_mnv3s"

SWEEP = [(0.5, 2.0), (0.7, 4.0), (0.9, 8.0)]   # (alpha, T)

## 1. Teacher

In [ ]:
teacher = build_model(TEACHER_NAME, num_classes)
print(TEACHER_NAME, "params: %.1f M" % (count_params(teacher) / 1e6))

hist_teacher = train_supervised(
    teacher, data, epochs=TEACHER_EPOCHS,
    head_lr=1e-3, backbone_lr=5e-5,
    ckpt_name="teacher_" + TEACHER_NAME, load_if_exists=True,
    tag="teacher/" + TEACHER_NAME,
)
res_teacher, test_teacher = evaluate_splits(teacher, data, desc="teacher")
report_both(res_teacher, test_teacher, class_names, title="teacher " + TEACHER_NAME)

## 2. Student baseline

In [ ]:
set_seed(SEED)
baseline = build_model(STUDENT_NAME, num_classes)
print(STUDENT_NAME, "params: %.2f M" % (count_params(baseline) / 1e6))

hist_base = train_supervised(
    baseline, data, epochs=STUDENT_EPOCHS,
    head_lr=2e-3, backbone_lr=5e-4,           # small nets tolerate (and need) a bigger lr
    ckpt_name=TAG + "_baseline", tag="baseline/" + STUDENT_NAME,
)
res_base, test_base = evaluate_splits(baseline, data, desc="baseline")
report_both(res_base, test_base, class_names, title="baseline " + STUDENT_NAME)

## 3. Sweep alpha and temperature

In [ ]:
sweep_results, sweep_hist, sweep_models = [], {}, {}

for alpha, T in SWEEP:
    set_seed(SEED)                                    # same init for every run
    s = build_model(STUDENT_NAME, num_classes)
    label = "a%.1f_T%.0f" % (alpha, T)
    h = train_kd(s, teacher, data, epochs=STUDENT_EPOCHS, alpha=alpha, T=T,
                 head_lr=2e-3, backbone_lr=5e-4,
                 ckpt_name="%s_%s" % (TAG, label), tag="KD " + label, verbose=False)
    v, t = evaluate_splits(s, data, desc=label)
    print("alpha=%.1f T=%.0f  ->  val acc %.4f F1 %.4f | test acc %.4f F1 %.4f"
          % (alpha, T, v["acc"], v["f1"], t["acc"], t["f1"]))
    sweep_results.append({"alpha": alpha, "T": T,
                          "val_acc": v["acc"], "val_f1": v["f1"],
                          "test_acc": t["acc"], "test_f1": t["f1"]})
    sweep_hist[label] = h
    sweep_models[label] = (s, v, t)

## 4. Compare

In [ ]:
# the winner is picked on VAL - picking it on test would spend the held-out set
best_label = max(sweep_models, key=lambda k: sweep_models[k][1]["f1"])
best_student, res_kd, test_kd = sweep_models[best_label]
print("best KD config (chosen on val):", best_label)

rows = summarize(
    [{"name": "teacher " + TEACHER_NAME,  "model": teacher,  "result": res_teacher, "test": test_teacher},
     {"name": "baseline " + STUDENT_NAME, "model": baseline, "result": res_base,    "test": test_base}] +
    [{"name": "KD " + k, "model": m, "result": v, "test": t}
     for k, (m, v, t) in sweep_models.items()]
)

print("\nbest-KD F1 gain over baseline: val %+.4f | test %+.4f"
      % (res_kd["f1"] - res_base["f1"], test_kd["f1"] - test_base["f1"]))
print("agreement with teacher on test: %.4f"
      % teacher_student_agreement(best_student, teacher, data["test_loader"]))

hist_all = {"baseline": hist_base}
hist_all.update(sweep_hist)
plot_histories(hist_all)
plot_confusion(test_kd, class_names, title="KD %s (%s, test)" % (STUDENT_NAME, best_label))
save_results(TAG, {"config": {"dataset": DATASET_DIR, "teacher": TEACHER_NAME,
                              "student": STUDENT_NAME, "sweep": SWEEP,
                              "teacher_epochs": TEACHER_EPOCHS, "student_epochs": STUDENT_EPOCHS,
                              "split": {"train": round(1 - VAL_SPLIT - TEST_SPLIT, 2),
                                        "val": VAL_SPLIT, "test": TEST_SPLIT}},
                   "best_on_val": best_label,
                   "sweep_results": sweep_results, "table": rows})